In [3]:
!pip install transformers

In [4]:
#!pip install sacrebleu

In [5]:
!pip install transformers[sentencepiece]

In [6]:
!pip install datasets
!pip install rouge_score
!pip install py7zr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 22.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 17.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 2

In [7]:
from transformers import pipeline, set_seed

import matplotlib.pyplot as plt

import pandas as pd
from datasets import load_dataset, load_metric
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer #class in the Hugging Face Transformers library provides a convenient way to automatically load the appropriate tokenizer for a given pre-trained model.

import pandas as pd
import numpy as np

import nltk
from nltk.tokenize import sent_tokenize

In [8]:
nltk.download("punkt") # data resource used by the NLTK library to perform sentence tokenization in various languages

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [9]:
from datasets import load_dataset

#dataset = load_dataset("cnn_dailymail", version="3.0.0")
dataset = load_dataset("cnn_dailymail", "3.0.0")

print(f"Features in cnn_dailymail : {dataset['train'].column_names}")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Features in cnn_dailymail : ['article', 'highlights', 'id']


In [10]:
# sample = dataset['train'][1]
# print(f"""
# Article (excerpt of 500 characters, total length: {len(sample["article"])}):
# """)
# print(sample["article"][:500])
# print(f'\nSummary (length: {len(sample["highlights"])}):')
# print(sample["highlights"])

# Extract the second sample from the 'train' subset of the dataset
sample = dataset['train'][1]

# Print a formatted string containing an excerpt of the article and its total length
print(f"""
Article (excerpt of 500 characters, total length: {len(sample["article"])}):
""")

# Print the first 500 characters of the article
print(sample["article"][:500])

# Print the length of the summary (highlights) of the article
print(f'\nSummary (length: {len(sample["highlights"])}):')

# Print the full summary (highlights) of the article
print(sample["highlights"])


Article (excerpt of 500 characters, total length: 4051):

Editor's note: In our Behind the Scenes series, CNN correspondents share their experiences in covering news and analyze the stories behind the events. Here, Soledad O'Brien takes users inside a jail where many of the inmates are mentally ill. An inmate housed on the "forgotten floor," where many mentally ill inmates are housed in Miami before trial. MIAMI, Florida (CNN) -- The ninth floor of the Miami-Dade pretrial detention facility is dubbed the "forgotten floor." Here, inmates with the most s

Summary (length: 281):
Mentally ill inmates in Miami are housed on the "forgotten floor"
Judge Steven Leifman says most are there as a result of "avoidable felonies"
While CNN tours facility, patient shouts: "I am the son of the president"
Leifman says the system is unjust and he's fighting for change .


In [11]:
sample_text = dataset['train'][1]["article"][:1000]

# We will collect the generated summaries of each model in a dictionay; initializes an empty dictionary
summaries = {}

In [12]:
def baseline_summary_three_sent(text):
    return "\n".join(sent_tokenize(text)[:3]) # taking the first three sentences of a given text

In [13]:
summaries['baseline'] = baseline_summary_three_sent(sample_text)
summaries['baseline']

'Editor\'s note: In our Behind the Scenes series, CNN correspondents share their experiences in covering news and analyze the stories behind the events.\nHere, Soledad O\'Brien takes users inside a jail where many of the inmates are mentally ill. An inmate housed on the "forgotten floor," where many mentally ill inmates are housed in Miami before trial.\nMIAMI, Florida (CNN) -- The ninth floor of the Miami-Dade pretrial detention facility is dubbed the "forgotten floor."'

## **GPT2**

In [14]:
from transformers import pipeline, set_seed

set_seed(42)
pipe = pipeline('text-generation', model = 'gpt2-medium')

gpt2_query = sample_text + "\nTL;DR:\n"

pipe_out = pipe(gpt2_query, max_length = 512, clean_up_tokenization_spaces = True) # clean up spaces near punctuation marks

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [15]:
pipe_out

[{'generated_text': 'Editor\'s note: In our Behind the Scenes series, CNN correspondents share their experiences in covering news and analyze the stories behind the events. Here, Soledad O\'Brien takes users inside a jail where many of the inmates are mentally ill. An inmate housed on the "forgotten floor," where many mentally ill inmates are housed in Miami before trial. MIAMI, Florida (CNN) -- The ninth floor of the Miami-Dade pretrial detention facility is dubbed the "forgotten floor." Here, inmates with the most severe mental illnesses are incarcerated until they\'re ready to appear in court. Most often, they face drug charges or charges of assaulting an officer --charges that Judge Steven Leifman says are usually "avoidable felonies." He says the arrests often result from confrontations with police. Mentally ill people often won\'t do what they\'re told when police arrive on the scene -- confrontation seems to exacerbate their illness and they become more paranoid, delusional, and

In [16]:
pipe_out[0]['generated_text'][len(gpt2_query) : ]

'To get to the jail that holds the mentally ill, visit our Behind the Scenes blog -- Click here The story doesn\'t end there:\xa0 In 2014, a judge ordered the jail to provide treatment for 40 mental-health detainees in the mental-health unit, as part of a $22,000 settlement.\nInmates in the mental-health unit at Miami-Dade County\'s jail are often locked in a cell that\'s usually just like any other in the facility and can become chaotic. Mental health unit employees are often required to leave the jail and drive to the facility, instead of coming in to work with the inmates themselves. Most mental-health detainees spend hours a day sleeping in front of the wall by the pool, waiting for treatment to show up.\nThere are two more stories from the Inside the Tincup Jail series:\n\xa0\xa0\xa0 The "no contact" policy:\xa0 At an overcrowded state mental health facility in Miami-Dade County, the rules allow the police to try to convince the jail staff to allow a person they believe is mentall

In [17]:
summaries['gpt2'] = "\n".join(sent_tokenize(pipe_out[0]['generated_text'][len(gpt2_query) : ]))

In [18]:
#t5

 ## **T5**

In [19]:
pipe = pipeline('summarization', model = 't5-small')
pipe_out = pipe(sample_text)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [20]:
pipe_out

[{'summary_text': "inmates with the most severe mental illnesses are incarcerated until they're ready to appear in court . most often, they face drug charges or charges of assaulting an officer . mentally ill people become more paranoid, delusional, and less likely to follow dir ."}]

In [21]:
summaries['t5'] = '\n'.join(sent_tokenize(pipe_out[0]['summary_text']))

In [22]:
# *BART*

## BART

In [23]:
pipe = pipeline("summarization", model = "facebook/bart-large-cnn")
pipe_out = pipe(sample_text)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [24]:
pipe_out

[{'summary_text': 'Miami-Dade pretrial detention facility is dubbed the "forgotten floor" Here, inmates with the most severe mental illnesses are incarcerated. Most often, they face drug charges or charges of assaulting an officer. Judge Steven Leifman says the arrests often result from confrontations with police.'}]

In [25]:
summaries['bart'] = "\n".join(sent_tokenize(pipe_out[0]["summary_text"]))

In [26]:
## PEGASUS

In [27]:
pipe = pipeline("summarization", model="google/pegasus-cnn_dailymail")
pipe_out = pipe(sample_text)

config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

In [28]:
pipe_out

[{'summary_text': 'Mentally ill inmates are housed on the "forgotten floor" of a Miami jail .<n>Judge Steven Leifman says the charges are usually "avoidable felonies"<n>He says the arrests often result from confrontations with police .<n>Mentally ill people often won\'t do what they\'re told when police arrive on the scene .'}]

In [29]:
summaries["pegasus"] = pipe_out[0]["summary_text"].replace(" .<n>", ".\n")

In [30]:
## comparing different summaries

In [31]:
print("Ground Truth")

print(dataset['train'][1]['highlights'])

for model_name in summaries:
    print(model_name.upper())
    print(summaries[model_name])

Ground Truth
Mentally ill inmates in Miami are housed on the "forgotten floor"
Judge Steven Leifman says most are there as a result of "avoidable felonies"
While CNN tours facility, patient shouts: "I am the son of the president"
Leifman says the system is unjust and he's fighting for change .
BASELINE
Editor's note: In our Behind the Scenes series, CNN correspondents share their experiences in covering news and analyze the stories behind the events.
Here, Soledad O'Brien takes users inside a jail where many of the inmates are mentally ill. An inmate housed on the "forgotten floor," where many mentally ill inmates are housed in Miami before trial.
MIAMI, Florida (CNN) -- The ninth floor of the Miami-Dade pretrial detention facility is dubbed the "forgotten floor."
GPT2
To get to the jail that holds the mentally ill, visit our Behind the Scenes blog -- Click here The story doesn't end there:  In 2014, a judge ordered the jail to provide treatment for 40 mental-health detainees in the me

In [32]:
## SacreBLEU

**SACREBLEU - not used now**

In [33]:
# from datasets import load_metric

# bleu_metric = load_metric("sacrebleu")

In [34]:

# import pandas as pd
# import numpy as np

# bleu_metric.add(
#     prediction="the the the the the the", reference=["the cat is on the mat"])
# results = bleu_metric.compute(smooth_method="floor", smooth_value=0)
# results["precisions"] = [np.round(p, 2) for p in results["precisions"]]
# pd.DataFrame.from_dict(results, orient="index", columns=["Value"])

In [35]:

# bleu_metric.add(
#     prediction="the cat is on mat", reference=["the cat is on the mat"])
# results = bleu_metric.compute(smooth_method="floor", smooth_value=0)
# results["precisions"] = [np.round(p, 2) for p in results["precisions"]]
# pd.DataFrame.from_dict(results, orient="index", columns=["Value"])

In [36]:
## Rouge

**ROUGE**

In [37]:
rouge_metric = load_metric('rouge')

<ipython-input-37-90542a62301a>:1: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  rouge_metric = load_metric('rouge')


The repository for rouge contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/rouge.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


In [38]:

reference = dataset["train"][1]["highlights"]
records = []
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

for model_name in summaries:
    rouge_metric.add(prediction=summaries[model_name], reference=reference)
    score = rouge_metric.compute()
    rouge_dict = dict((rn, score[rn].mid.fmeasure) for rn in rouge_names)
    records.append(rouge_dict)
pd.DataFrame.from_records(records, index=summaries.keys())

,rouge1,rouge2,rougeL,rougeLsum
baseline,0.365079,0.145161,0.206349,0.285714
gpt2,0.157895,0.026490,0.111842,0.144737
t5,0.197802,0.022472,0.131868,0.197802
bart,0.365591,0.131868,0.215054,0.322581
pegasus,0.500000,0.244898,0.360000,0.460000


In [39]:
## Evaluating on the Test set of the CNN Daily Mail Dataset

**Evaluating on the Test set of the CNN Daily Mail Dataset**

In [40]:
def calculate_metric_on_baseline_test_ds(dataset, metric, column_text = 'article', column_summary = 'highlights' ):
    summaries = [baseline_summary_three_sent(text) for text in dataset[column_text] ]

    metric.add_batch(predictions = summaries, references = dataset[column_summary] )

    score = metric.compute()
    return score

In [41]:
test_sampled = dataset['train'].shuffle(seed = 42).select(range(1000))

score = calculate_metric_on_baseline_test_ds(test_sampled, rouge_metric )

rouge_dict = dict((rn, score[rn].mid.fmeasure ) for rn in rouge_names )

pd.DataFrame.from_dict(rouge_dict, orient = 'index' , columns = ['baseline'] ).T

,rouge1,rouge2,rougeL,rougeLsum
baseline,0.253995,0.100642,0.165754,0.231571


In [42]:
## Strategy to calculate the Rouge Metric on test dataset for the other models


## **Calculate the Rouge Metric on test dataset**


In [43]:
from tqdm import tqdm
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

def generate_batch_sized_chunks(list_of_elements, batch_size):
    """split the dataset into smaller batches that we can process simultaneously
    Yield successive batch-sized chunks from list_of_elements."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                               batch_size=16, device=device,
                               column_text="article",
                               column_summary="highlights"):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)):

        inputs = tokenizer(article_batch, max_length=1024,  truncation=True,
                        padding="max_length", return_tensors="pt")

        summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                         attention_mask=inputs["attention_mask"].to(device),
                         length_penalty=0.8, num_beams=8, max_length=128)
        ''' parameter for length penalty ensures that the model does not generate sequences that are too long. '''

        # Finally, we decode the generated texts,
        # replace the <n> token, and add the decoded texts with the references to the metric.
        decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                               clean_up_tokenization_spaces=True)
               for s in summaries]

        decoded_summaries = [d.replace("<n>", " ") for d in decoded_summaries]


        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    #  Finally compute and return the ROUGE scores.
    score = metric.compute()
    return score

In [44]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_ckpt = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
# Define the device: use GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

score = calculate_metric_on_test_ds(test_sampled, rouge_metric,
                                   model_pegasus, tokenizer, batch_size=8)

rouge_dict = dict((rn, score[rn].mid.fmeasure) for rn in rouge_names)

# At the end, we compute and return the ROUGE scores.
pd.DataFrame(rouge_dict, index=["pegasus"])

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 125/125 [23:24<00:00, 11.24s/it]


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.47518,0.278443,0.37406,0.428993


In [45]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import pandas as pd

# Define the device to use GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Specify the model checkpoint for GPT-2 Medium
model_ckpt = "gpt2-medium"

# Load the tokenizer and model for GPT-2 Medium
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model_gpt2 = AutoModelForCausalLM.from_pretrained(model_ckpt).to(device)

def generate_text_gpt2(input_texts, model, tokenizer, max_length=50):
    inputs = tokenizer(input_texts, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model.generate(inputs['input_ids'], max_length=max_length, num_return_sequences=1)
    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return generated_texts

def calculate_metric_on_test_ds_gpt2(test_dataset, rouge_metric, model, tokenizer, batch_size=8):
    preds, refs = [], []
    for i in range(0, len(test_dataset), batch_size):
        batch = test_dataset[i:i + batch_size]
        inputs = [item["input_text"] for item in batch]
        references = [item["reference_text"] for item in batch]
        predictions = generate_text_gpt2(inputs, model, tokenizer)
        preds.extend(predictions)
        refs.extend(references)
    result = rouge_metric.compute(predictions=preds, references=refs)
    return result

# Calculate the ROUGE scores on the test dataset
score = calculate_metric_on_test_ds_gpt2(test_sampled, rouge_metric, model_gpt2, tokenizer, batch_size=8)

rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_dict = dict((rn, score[rn].mid.fmeasure) for rn in rouge_names)

# Compute and return the ROUGE scores as a DataFrame
pd.DataFrame(rouge_dict, index=["gpt2-medium"])


TypeError: string indices must be integers

In [1]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_ckpt = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

score = calculate_metric_on_test_ds(test_sampled, rouge_metric,
                                   model_pegasus, tokenizer, batch_size=8)

rouge_dict = dict((rn, score[rn].mid.fmeasure) for rn in rouge_names)

# At the end, we compute and return the ROUGE scores.
pd.DataFrame(rouge_dict, index=["T5"])

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

NameError: name 'device' is not defined

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
import pandas as pd

# Define the device to use GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Specify the model checkpoint for T5 Small
model_ckpt = "t5-small"

# Load the tokenizer and model for T5 Small
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

def generate_text_t5(input_texts, model, tokenizer, max_length=50):
    inputs = tokenizer(input_texts, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model.generate(inputs['input_ids'], max_length=max_length, num_return_sequences=1)
    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return generated_texts

def calculate_metric_on_test_ds_t5(test_dataset, rouge_metric, model, tokenizer, batch_size=8):
    preds, refs = [], []
    for i in range(0, len(test_dataset), batch_size):
        batch = test_dataset[i:i + batch_size]
        inputs = [item["input_text"] for item in batch]
        references = [item["reference_text"] for item in batch]
        predictions = generate_text_t5(inputs, model, tokenizer)
        preds.extend(predictions)
        refs.extend(references)
    result = rouge_metric.compute(predictions=preds, references=refs)
    return result

# Calculate the ROUGE scores on the test dataset
score = calculate_metric_on_test_ds_t5(test_sampled, rouge_metric, model_t5, tokenizer, batch_size=8)

rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_dict = dict((rn, score[rn].mid.fmeasure) for rn in rouge_names)

# Compute and return the ROUGE scores as a DataFrame
rouge_df = pd.DataFrame(rouge_dict, index=["t5-small"])
print(rouge_df)


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
import pandas as pd

# Define the device to use GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Specify the model checkpoint for BART
model_ckpt = "facebook/bart-large"

# Load the tokenizer and model for BART
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model_bart = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

def calculate_metric_on_test_ds(test_dataset, rouge_metric, model, tokenizer, batch_size=8):
    preds, refs = [], []
    for i in range(0, len(test_dataset), batch_size):
        batch = test_dataset[i:i + batch_size]
        inputs = [item["input_text"] for item in batch]
        references = [item["reference_text"] for item in batch]
        # Tokenize and generate predictions
        inputs = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            outputs = model.generate(inputs['input_ids'], max_length=50, num_return_sequences=1)
        predictions = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
        preds.extend(predictions)
        refs.extend(references)
    # Compute ROUGE scores
    result = rouge_metric.compute(predictions=preds, references=refs)
    return result

# Calculate the ROUGE scores on the test dataset
score = calculate_metric_on_test_ds(test_sampled, rouge_metric, model_bart, tokenizer, batch_size=8)

rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_dict = dict((rn, score[rn].mid.fmeasure) for rn in rouge_names)

# Compute and return the ROUGE scores as a DataFrame
rouge_df = pd.DataFrame(rouge_dict, index=["BART"])
print(rouge_df)


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
import pandas as pd

# Define the device to use GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Specify the model checkpoint for BART
model_ckpt = "facebook/bart-large"

# Load the tokenizer and model for BART
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model_bart = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

def calculate_metric_on_test_ds(test_dataset, rouge_metric, model, tokenizer, batch_size=8):
    preds, refs = [], []
    for i in range(0, len(test_dataset), batch_size):
        batch = test_dataset[i:i + batch_size]
        inputs = [item["input_text"] for item in batch]
        references = [item["reference_text"] for item in batch]
        # Tokenize and generate predictions
        inputs = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            outputs = model.generate(inputs['input_ids'], max_length=50, num_return_sequences=1)
        predictions = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
        preds.extend(predictions)
        refs.extend(references)
    # Compute ROUGE scores
    result = rouge_metric.compute(predictions=preds, references=refs)
    return result

# Calculate the ROUGE scores on the test dataset
score = calculate_metric_on_test_ds(test_sampled, rouge_metric, model_bart, tokenizer, batch_size=8)

rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_dict = dict((rn, score[rn].mid.fmeasure) for rn in rouge_names)

# Compute and return the ROUGE scores as a DataFrame
rouge_df = pd.DataFrame(rouge_dict, index=["BART"])
print(rouge_df)
